# Train the pattern classifier
In this notebook, a neural network is trained to identify the pattern type of the initial vortex data given the vorticity at a later time.

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load data

The classifier is trained on the evolved vorticity fields $\omega(T)$, or `omegaT`, generated for each of the four vortex families: same sign pairs, opposite sign pairs, same sign triples, and mixed sign triples. Each family contains 20,000 samples.

For classification, only the evolved field `omegaT` is used as input. Each sample is assigned an integer label identifying its vortex family, and the four datasets are then combined into a single balanced classification dataset.

In [3]:
files = {
    "same pair": "vortex_same_sign_pair_20000_aug.npz",
    "opposite pair": "vortex_opposite_sign_pair_20000_aug.npz",
    "same triple": "vortex_same_sign_triple_20000_aug.npz",
    "mixed triple": "vortex_mixed_sign_triple_20000_aug.npz",
}

label_to_id = {
    "same pair": 0,
    "opposite pair": 1,
    "same triple": 2,
    "mixed triple": 3,
}

Xs = []
ys = []

for label_name, filename in files.items():
    data = np.load(filename)

    print(filename)
    print(data.files)

    X = data["omegaT"]

    y = np.full(
        X.shape[0],
        label_to_id[label_name],
        dtype=np.int64,
    )
    print(X.shape)
    Xs.append(X)
    ys.append(y)

X_all = np.concatenate(Xs)
y_all = np.concatenate(ys)

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)
print("label counts:", np.bincount(y_all))

vortex_same_sign_pair_20000_aug.npz
['omega0', 'parameters', 'active_vortex_mask', 'vortex_count', 'pattern_name', 'omegaT']
(20000, 128, 128)
vortex_opposite_sign_pair_20000_aug.npz
['omega0', 'parameters', 'active_vortex_mask', 'vortex_count', 'pattern_name', 'omegaT']
(20000, 128, 128)
vortex_same_sign_triple_20000_aug.npz
['omega0', 'parameters', 'active_vortex_mask', 'vortex_count', 'pattern_name', 'omegaT']
(20000, 128, 128)
vortex_mixed_sign_triple_20000_aug.npz
['omega0', 'parameters', 'active_vortex_mask', 'vortex_count', 'pattern_name', 'omegaT']
(20000, 128, 128)
X_all shape: (80000, 128, 128)
y_all shape: (80000,)
label counts: [20000 20000 20000 20000]


# Preparing data for training

The vorticity fields are converted to PyTorch tensors and given an additional channel dimension, as required by `Conv2d`. The complete dataset is then divided into an 80% training set and a 20% test set. Samples are supplied to the network in batches of 64, with the training data shuffled between epochs.

In [3]:
# We add a dimension to X_all since Conv2d expects a dimension specifically for channel.
X_all = X_all[:, None, :, :]

# Now we convert data to torch tensors.
X_t = torch.tensor(X_all, dtype=torch.float32)
y_t = torch.tensor(y_all, dtype=torch.long)

In [4]:
# We divide the dataset into training data and testing data, in batches.

dataset = TensorDataset(X_t, y_t)

num_total = len(dataset)
num_train = int(0.8*num_total)
num_test = num_total - num_train

train_dataset, test_dataset = random_split(
    dataset,
    [num_train, num_test],
    generator=torch.Generator().manual_seed(0),
)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
)

print("num_train:", len(train_dataset))
print("num_test:", len(test_dataset))

num_train: 64000
num_test: 16000


# Pattern classifier

The classifier is a convolutional neural network that takes a two-dimensional evolved vorticity field as input and outputs four scores, one for each vortex family. Convolutional layers extract spatial features from the field while successive pooling operations reduce the spatial resolution. Circular padding is used in the convolutional layers to respect the periodic domain we have chosen for the problem.

After the convolutional stages, the extracted features are flattened and passed through fully connected layers to produce the four final class scores.

In [4]:
# Now we define our convolutional neural network.
# Since there are four different pattern types (same sign pair, opposite sign pair, same sign triple, mixed sign triple),
# for each input omegaT, the model outputs four scores, one for each possible pattern.

model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=5, padding=2, padding_mode="circular"),
    nn.Tanh(),

    nn.Conv2d(8, 16, kernel_size=5, padding=2, padding_mode="circular"),
    nn.Tanh(),

    nn.AvgPool2d(kernel_size=2),

    nn.Conv2d(16, 32, kernel_size=5, padding=2, padding_mode="circular"),
    nn.Tanh(),

    nn.AvgPool2d(kernel_size=2),

    nn.Conv2d(32, 64, kernel_size=5, padding=2, padding_mode="circular"),
    nn.Tanh(),

    nn.AdaptiveAvgPool2d((8, 8)),

    nn.Flatten(),

    nn.Linear(64 * 8 * 8, 64),
    nn.Tanh(),

    nn.Linear(64, 4),
).to(device)

# Training

The network is trained using cross-entropy loss, which compares the four output class scores with the true vortex family label. Model parameters are updated with the Adam optimizer using a learning rate of $3\times 10^{-4}$.

At each epoch, both the loss and classification accuracy are computed on the training set and on the test set. The test set is evaluated without gradient calculations and is not used to update the model.

In [6]:
# Now we train the model

loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=3e-4)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()

    total_train_loss = 0.0
    total_train_correct = 0
    total_train_num = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)
        loss = loss_fn(logits, yb)

        loss.backward()
        optimizer.step()

        current_batch_size = xb.shape[0]

        total_train_loss += loss.item() * current_batch_size

        preds = logits.argmax(dim=1)
        total_train_correct += (preds == yb).sum().item()
        total_train_num += current_batch_size

    train_loss = total_train_loss / total_train_num
    train_acc = total_train_correct / total_train_num

    # Evaluate on test set
    model.eval()

    total_test_loss = 0.0
    total_test_correct = 0
    total_test_num = 0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = loss_fn(logits, yb)

            current_batch_size = xb.shape[0]

            total_test_loss += loss.item() * current_batch_size

            preds = logits.argmax(dim=1)
            total_test_correct += (preds == yb).sum().item()
            total_test_num += current_batch_size

    test_loss = total_test_loss / total_test_num
    test_acc = total_test_correct / total_test_num

    print(
        f"epoch {epoch:3d} | "
        f"train loss {train_loss:.4f} | train accuracy {train_acc:.4f} | "
        f"test loss {test_loss:.4f} | test accuracy {test_acc:.4f}"
    )

epoch   0 | train loss 0.9566 | train accuracy 0.5517 | test loss 0.3749 | test accuracy 0.8589
epoch   1 | train loss 0.2244 | train accuracy 0.9132 | test loss 0.1224 | test accuracy 0.9589
epoch   2 | train loss 0.0961 | train accuracy 0.9665 | test loss 0.0587 | test accuracy 0.9801
epoch   3 | train loss 0.0382 | train accuracy 0.9881 | test loss 0.0309 | test accuracy 0.9905
epoch   4 | train loss 0.0325 | train accuracy 0.9893 | test loss 0.0116 | test accuracy 0.9976
epoch   5 | train loss 0.0266 | train accuracy 0.9919 | test loss 0.0262 | test accuracy 0.9931
epoch   6 | train loss 0.0152 | train accuracy 0.9956 | test loss 0.0088 | test accuracy 0.9981
epoch   7 | train loss 0.0138 | train accuracy 0.9958 | test loss 0.0040 | test accuracy 0.9989
epoch   8 | train loss 0.0124 | train accuracy 0.9960 | test loss 0.0051 | test accuracy 0.9994
epoch   9 | train loss 0.0109 | train accuracy 0.9965 | test loss 0.0034 | test accuracy 0.9992


# Training results

The classifier rapidly learns the four vortex families. Test accuracy rises from about 86% to above 99% after four epochs, eventually reaching approximately 99.9%. Training and test performance remain close throughout training, indicating that the classifier generalizes well to non-training data.

## Save trained classifier

In [7]:
# Save the pattern classifier model
torch.save(model.state_dict(), "pattern_classifier.pt")